# Agent 3 Mechanical Expert 워크플로우 테스트

이 노트북은 `Mechanical Expert` (기계적 손상 전문가)의 독립적인 워크플로우를 테스트합니다.

## 주요 기능
- 테스트 이미지 로드
- Mechanical Expert 그래프 실행 (Step 1 -> Step 2 -> Step 3)
- 결과 분석 및 리포트 출력

In [1]:
# 1. 초기 설정 및 라이브러리 임포트
import sys
import os
from pathlib import Path
import json
import base64

# 프로젝트 루트 경로 설정
project_root = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
sys.path.insert(0, str(project_root))

# 환경 변수 로드
from dotenv import load_dotenv
load_dotenv(project_root / ".env")

print(f"✓ 프로젝트 루트: {project_root}")

✓ 프로젝트 루트: c:\Users\loidn\Documents\Projects\P_04_Scope


In [2]:
# 2. 테스트 이미지 준비
from src.utils import find_data_directory

try:
    data_dir = find_data_directory()
    test_image_name = "IMG_8113.jpg"
    test_image_path = Path(data_dir) / test_image_name
    
    if not test_image_path.exists():
        # 이미지가 없으면 첫 번째 가능한 이미지 사용
        image_files = list(Path(data_dir).glob("*.png")) + list(Path(data_dir).glob("*.jpg"))
        if image_files:
            test_image_path = image_files[0]
            print(f"⚠️ {test_image_name}을 찾을 수 없어 {test_image_path.name}을 사용합니다.")
        else:
            raise FileNotFoundError("테스트할 이미지가 없습니다.")
            
    print(f"✓ 테스트 이미지: {test_image_path}")
except Exception as e:
    print(f"❌ 오류: {e}")

✓ 테스트 이미지: c:\Users\loidn\Documents\Projects\P_04_Scope\data\IMG_8113.jpg


In [3]:
# 3. Payload 생성 함수
def create_payload(image_path):
    with open(image_path, 'rb') as f:
        image_data = f.read()
    
    ext = Path(image_path).suffix.lower()
    mime_type = 'image/png' if ext == '.png' else 'image/jpeg'
    image_base64 = base64.b64encode(image_data).decode('utf-8')
    
    return [
        {"text": "이미지를 분석하세요."},
        {"inline_data": {"mime_type": mime_type, "data": image_base64}}
    ]

In [4]:
# 4. Mechanical Expert 실행
from src.graphs.mechanical_expert_graph import mechanical_expert_wrapper_node
from src.state import InvestigationState

print("Mechanical Expert 분석 시작...")

payload = create_payload(test_image_path)

# InvestigationState 모의 구성
mock_state = InvestigationState(
    payload=payload,
    expert_reports=[],
    expert_analysis_results={},
    expert_confidence_scores={},
    expert_evidence={},
    final_verdict=None,
    errors=[],
    mechanical_cached_image_data=None
)

try:
    result = mechanical_expert_wrapper_node(mock_state)
    
    print("\n✓ 분석 완료!")
    print("=" * 60)
    
    # 결과 출력
    reports = result.get("expert_reports", [])
    if reports:
        print(reports[0])
    else:
        print("리포트가 생성되지 않았습니다.")
        
    print("=" * 60)
    print("세부 분석 결과:")
    analysis_results = result.get("expert_analysis_results", {}).get("mechanical", {})
    print(json.dumps(analysis_results, indent=2, ensure_ascii=False))
    
except Exception as e:
    print(f"❌ 실행 중 오류 발생: {e}")
    import traceback
    traceback.print_exc()

Mechanical Expert 분석 시작...

==================== Agent Reasoning & Tool Execution ====================
🛠️ [Tool Call]: apply_clahe_filter (Args: {'image_path': 'C:\\Users\\loidn\\AppData\\Local\\Temp\\tmp18fvp4f6.jpg'})
   └─ 📊 [Tool Output]: CLAHE 필터 적용 완료
- 이미지 크기: 1568x1176
🛠️ [Tool Call]: analyze_mechanical_deformation_internal (Args: {'image_path': 'C:\\Users\\loidn\\AppData\\Local\\Temp\\tmp18fvp4f6.jpg'})
   └─ 📊 [Tool Output]: {"mechanical_deformation_detected": false, "deformation_type": "unknown", "deformation_location": "전선 끝단 및 용융 부위 주변 (명확한 기계적 변형 부위 없음)", "deformation_on_non_melted_part": false, "tool_marks_detected": false, "tool_mark_description": "도구에 의한 규칙적인 패턴이나 찍힘 흔적이 관찰되지 않음", "arc_bead_proximity": "전선 끝단에 용융...

🧠 [Thought]:
Final Answer: 분석 결과, 해당 이미지에서 **기계적 변형 흔적은 식별되지 않았습니다.** 상세 분석 내용은 다음과 같습니다.

1. **기계적 변형 및 도구 흔적**: 플라이어나 절단기 등에 의한 찍힘, 압착, 규칙적인 절단면과 같은 인위적인 냉간 변형 흔적이 관찰되지 않았습니다.
2. **주요 관찰 사항**: 전선 끝단의 연선들이 흩어져 있으며 일부 용융된 흔적이 확인됩니다. 이는 기계적 손상보다는 과전류나 외부 열원

In [ ]:
# 5. 분석 결과 시각화 (AI 검출 위치 표시)
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import io
import numpy as np

def draw_bounding_boxes(image_path, analysis_results):
    """
    분석 결과에 포함된 Bounding Box를 이미지 위에 표시합니다.
    """
    try:
        # 이미지 로드
        img = Image.open(image_path)
        original_width, original_height = img.size
        
        fig, ax = plt.subplots(figsize=(12, 12))
        ax.imshow(img)
        ax.set_title("AI Detected Regions", fontsize=15)
        axis_off = True
        
        colors = {'step1': 'red', 'step2': 'orange', 'step3': 'blue', 'step4': 'green'}
        labels = {'step1': 'Step 1', 'step2': 'Step 2', 'step3': 'Step 3', 'step4': 'Step 4'}
        
        legend_patches = []
        
        # 각 단계별 결과 시각화
        for step_key, color in colors.items():
            step_result = analysis_results.get(step_key, {})
            bboxes = step_result.get('bboxes', [])
            
            if bboxes:
                axis_off = False
                legend_patches.append(patches.Patch(color=color, label=f"{labels[step_key]} ({len(bboxes)})"))
                
                for box in bboxes:
                    # 0-1000 정규화 좌표 -> 픽셀 좌표 변환
                    # box format: [ymin, xmin, ymax, xmax]
                    ymin, xmin, ymax, xmax = box
                    
                    x = xmin / 1000 * original_width
                    y = ymin / 1000 * original_height
                    w = (xmax - xmin) / 1000 * original_width
                    h = (ymax - ymin) / 1000 * original_height
                    
                    # 사각형 그리기
                    rect = patches.Rectangle((x, y), w, h, linewidth=2, edgecolor=color, facecolor='none')
                    ax.add_patch(rect)

        if legend_patches:
            ax.legend(handles=legend_patches, loc='upper right')
            
        plt.axis('off')
        plt.show()
        
        if not legend_patches:
            print("검출된 Bounding Box 정보가 없습니다.")
            
    except Exception as e:
        print(f"시각화 중 오류 발생: {e}")

# 시각화 함수 실행
if 'test_image_path' in locals() and 'result' in locals():
    # 결과에서 전문가 키 찾기 (동적)
    expert_results = result.get("expert_analysis_results", {})
    target_key = next(iter(expert_results), None) # 첫 번째 키 사용
    
    if target_key:
        print(f"Visualizing results for expert: {target_key}")
        analysis_results = expert_results[target_key]
        draw_bounding_boxes(test_image_path, analysis_results)
    else:
        print("분석 결과(expert_analysis_results)가 비어 있습니다.")
else:
    print("테스트 이미지 경로 또는 분석 결과가 없습니다.")